In [ ]:
# Library imports
import os
import pandas as pd
import datetime
import openpyxl

# Create Outputs directory if it doesn't exist
output_dir = "Outputs"
os.makedirs(output_dir, exist_ok=True)

# File paths and names mapping (for better readability in output)
file_mapping = {
    "adv_mt": {
        "path": "AMT.xlsx",
        "sheet": "AMT",
        "display_name": "AMT"
    },
    "adv_st": {
        "path": "Navigator.xlsx",
        "sheet": "Navigator",
        "display_name": "Navigator"
    }, 
    "DEET_ST": {
        "path": "DEET.xlsx",
        "sheet": "DEET",
        "display_name": "DEET"
    },
    "Nebula_Galaxy": {
        "path": "Nebula_Galaxy.xlsx",
        "sheet": "NebGal",
        "display_name": "Nebula-Galaxy"
    },
    "RB": {
        "path": "Railblazer.xlsx",
        "sheet": "ServiceWatch Log Parameters",
        "display_name": "RB"
    }
}




In [ ]:
def merge_finalist_files():
    try:
        # Create Final Output directory if it doesn't exist
        final_output_dir = "Final Output"
        os.makedirs(final_output_dir, exist_ok=True)
        
        # Path to Finalists folder
        finalists_dir = os.path.join(output_dir, "Finalists")
        
        # Check if Finalists directory exists
        if not os.path.exists(finalists_dir):
            print(f"Error: Finalists directory not found at {finalists_dir}")
            return
            
        # List all Excel files in the Finalists directory, excluding temp files
        excel_files = [f for f in os.listdir(finalists_dir) 
                      if f.endswith('.xlsx') and not f.startswith('~$')]
        
        if not excel_files:
            print(f"No Excel files found in {finalists_dir}")
            return
            
        # Initialize an empty DataFrame to store merged data
        merged_df = pd.DataFrame()
        
        # Read and merge each Excel file
        for file in excel_files:
            file_path = os.path.join(finalists_dir, file)
            
            # Determine platform based on filename
            platform = None
            if 'Navigator' in file:
                platform = 'Navigator'
            elif 'DEET' in file:
                platform = 'DEET'
            elif 'Nebula_Galaxy' in file:
                platform = 'Nebula-Galaxy'
            elif 'Railblazer' in file:
                platform = 'Railblazer'
            elif 'AMT' in file:
                platform = 'AMT'
            else:
                platform = 'Unknown'
            
            try:
                # Read the Excel file
                df = pd.read_excel(file_path)
                
                # Check for duplicate columns and keep only one instance
                # This gets a list of columns with duplicated names
                duplicated_cols = [col for col in df.columns if df.columns.tolist().count(col) > 1]
                
                # If duplicates exist, keep only the first occurrence
                if duplicated_cols:
                    print(f"Found duplicate columns in {file}: {set(duplicated_cols)}")
                    # Get unique column names while preserving order
                    unique_cols = []
                    for col in df.columns:
                        if col not in unique_cols:
                            unique_cols.append(col)
                    
                    # Select only unique columns
                    df = df[unique_cols]
                
                # Add Platform column
                df['Platform'] = platform
                
                # Append to merged DataFrame
                if merged_df.empty:
                    merged_df = df.copy()
                else:
                    # Ensure column order is consistent before merging
                    # This is important to avoid duplicate columns with different positions
                    common_cols = list(set(merged_df.columns) & set(df.columns))
                    new_cols = [col for col in df.columns if col not in merged_df.columns]
                    
                    # Reorder columns in the new dataframe to match the merged one
                    if common_cols:
                        df = df[common_cols + new_cols]
                    
                    merged_df = pd.concat([merged_df, df], ignore_index=True)
                
                print(f"Added {len(df)} rows from {file} ({platform})")
                
            except Exception as e:
                print(f"Error reading {file}: {str(e)}")
            
        if merged_df.empty:
            print("No data was found in the Excel files.")
            return
        
        # Move Platform column to the front
        if 'Platform' in merged_df.columns:
            # Get all columns except Platform
            other_cols = [col for col in merged_df.columns if col != 'Platform']
            # Reorder columns with Platform first
            merged_df = merged_df[['Platform'] + other_cols]
        
        # Save the merged DataFrame to a new Excel file
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = os.path.join(final_output_dir, f'Merged_ServiceWatch_Parameters_{timestamp}.xlsx')
        merged_df.to_excel(output_file, index=False)
        
        print(f"\nMerging complete. Output saved to: {output_file}")
        print(f"Total rows: {len(merged_df)}")
        print(f"Total columns: {len(merged_df.columns)}")
        print(f"Column names: {', '.join(merged_df.columns)}")
        
    except Exception as e:
        print(f"Error merging finalist files: {str(e)}")

# Execute the function
merge_finalist_files()

Added 172 rows from AMT_Analysis_Final.xlsx (AMT)
Added 100 rows from DEET_Analysis_Final.xlsx (DEET)
Added 100 rows from Navigator_Analysis_Final.xlsx (Navigator)
Added 164 rows from Nebula_Galaxy_Analysis_Final.xlsx (Nebula-Galaxy)
Added 115 rows from Railblazer_Analysis_Final.xlsx (Railblazer)

Merging complete. Output saved to: Final Output\Merged_ServiceWatch_Parameters_20250807_175920.xlsx
Total rows: 651
Total columns: 16
Column names: SPN, User Display Name, Display Types, TIME, Event, Config, Dependency, Access Level (Wintrac), Unit of Measurement, Event log when alarm code(s) set, Event Log When Parameter Changes?, Auto Log at Power ON?, Auto Log at Noon (12:05 PM)?, Additional Reqs, Documentation Notes, Platform


In [23]:
def process_navigator_sheet():
    try:
        # Read Navigator Excel file
        nav_file = file_mapping["adv_st"]["path"]
        
        # Read Excel file with openpyxl
        wb = openpyxl.load_workbook(nav_file)
        ws = wb['Navigator']
        
        # Create a list to store non-strikethrough row indices
        rows_to_keep = []
        
        # Check each row for strikethrough formatting
        for row_idx, row in enumerate(ws.rows):
            # Check first cell in each row
            cell = row[0]
            
            # If the font doesn't have strikethrough or cell is empty, keep the row
            if not cell.font.strike or cell.value is None:
                rows_to_keep.append(row_idx)
        
        # Read the Excel file into DataFrame, keeping only non-strikethrough rows
        nav_df = pd.read_excel(nav_file, sheet_name="Navigator", skiprows=lambda x: x not in rows_to_keep)
        
        event_idx = nav_df.columns.get_loc('Event')
        nav_df.insert(event_idx + 1, 'Config', False)

        alarm_idx = nav_df.columns.get_loc('Event log when alarm code(s) set')
        new_columns = [
            'Event Log When Parameter Changes?',
            'Auto Log at Power ON?',
            'Auto Log at Noon (12:05 PM)?'
        ]
        
        # Initialize new columns with 'FALSE'
        for idx, col in enumerate(reversed(new_columns)):
            nav_df.insert(alarm_idx + 1, col, '')

        # Process Additional Reqs column
        for idx, row in nav_df.iterrows():
            additional_reqs = str(row['Additional Reqs']).lower()  # Convert to string and lowercase for better matching
            
            # Check for noon logging
            if any(keyword in additional_reqs for keyword in ['12:05', 'midday', 'noon']):
                nav_df.at[idx, 'Auto Log at Noon (12:05 PM)?'] = 'TRUE'
            
            # Check for power on/off logging
            if any(keyword in additional_reqs for keyword in ['power on', 'power off', 'power-on', 'power-off', 'poweron', 'poweroff']):
                nav_df.at[idx, 'Auto Log at Power ON?'] = 'TRUE'
            
            # Check for parameter changes logging
            if any(keyword in additional_reqs for keyword in ['parameter changes', 'param changes', 'value change', 'changes']):
                nav_df.at[idx, 'Event Log When Parameter Changes?'] = 'TRUE'

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = os.path.join(output_dir, f'Navigator_Analysis_{timestamp}.xlsx')

        nav_df.to_excel(output_file, index=False)
        print(f"Processing complete. Output saved to: {output_file}")
        print(f"Removed {ws.max_row - len(rows_to_keep)} strikethrough rows")

    except Exception as e:
        print(f"Error processing Navigator sheet: {str(e)}")

# Execute the function
process_navigator_sheet()

Processing complete. Output saved to: Outputs\Navigator_Analysis_20250807_170313.xlsx
Removed 1 strikethrough rows


In [22]:
def process_deet_sheet():
    try:
        # Read DEET Excel file
        deet_file = file_mapping["DEET_ST"]["path"]
        
        # Read Excel file with openpyxl
        wb = openpyxl.load_workbook(deet_file)
        ws = wb['DEET']
        
        # Create a list to store non-strikethrough row indices
        rows_to_keep = []
        
        # Check each row for strikethrough formatting
        for row_idx, row in enumerate(ws.rows):
            cell = row[0]
            if not cell.font.strike or cell.value is None:
                rows_to_keep.append(row_idx)
        
        # Read the Excel file into DataFrame
        deet_df = pd.read_excel(deet_file, sheet_name="DEET", skiprows=lambda x: x not in rows_to_keep)
        
        # Filter rows based on Platform column
        deet_df = deet_df[deet_df['Platform'].str.contains('DEET|Navigator/DEET', case=False, na=False)]
        
        # Add columns after "Default Log"
        default_log_idx = deet_df.columns.get_loc('Default Log')
        new_columns = ['TIME', 'Event', 'Config']
        
        # Insert columns after Default Log with default FALSE values
        for idx, col in enumerate(reversed(new_columns)):
            deet_df.insert(default_log_idx + 1, col, 'FALSE')
        
        # Process Default Log column
        for idx, row in deet_df.iterrows():
            default_log = str(row['Default Log']).lower()  # Convert to string and lowercase for better matching
            
            # Check for Timed/Periodic logging
            if any(keyword in default_log for keyword in ['timed', 'periodic']):
                deet_df.at[idx, 'TIME'] = 'TRUE'
            
            # Check for Event logging
            if 'event' in default_log:
                deet_df.at[idx, 'Event'] = 'TRUE'
            
            # Check for Configuration logging
            if 'reefer configuration' in default_log:
                deet_df.at[idx, 'Config'] = 'TRUE'
        
        # Add columns after "Event log when alarm code(s) set"
        alarm_idx = deet_df.columns.get_loc('Event log when alarm code(s) set')
        log_columns = [
            'Event Log When Parameter Changes?',
            'Auto Log at Power ON?',
            'Auto Log at Noon (12:05 PM)?'
        ]
        
        # Insert columns after Event log when alarm code(s) set with default FALSE values
        for idx, col in enumerate(reversed(log_columns)):
            deet_df.insert(alarm_idx + 1, col, '')

        # Process Additional Reqs column
        for idx, row in deet_df.iterrows():
            additional_reqs = str(row['Additional Reqs']).lower()  # Convert to string and lowercase for better matching
            
            # Check for noon/midday logging
            if any(keyword in additional_reqs for keyword in ['12:05', 'midday', 'noon', 'middle of day']):
                deet_df.at[idx, 'Auto Log at Noon (12:05 PM)?'] = 'TRUE'
            
            # Check for power on/startup logging
            if any(keyword in additional_reqs for keyword in ['power on', 'power-on', 'poweron', 'start up', 'startup', 'power cycle']):
                deet_df.at[idx, 'Auto Log at Power ON?'] = 'TRUE'
            
            # Check for parameter/value changes logging
            if any(keyword in additional_reqs for keyword in ['parameter', 'param', 'value change', 'state change', 'changes']):
                deet_df.at[idx, 'Event Log When Parameter Changes?'] = 'TRUE'

        # ...rest of existing code for saving file...

        # Remove Platform and Default Log columns
        deet_df = deet_df.drop(['Platform', 'Default Log'], axis=1)

        # Create output directory and save file
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = os.path.join(output_dir, f'DEET_Analysis_{timestamp}.xlsx')
        deet_df.to_excel(output_file, index=False)
        
        print(f"Processing complete. Output saved to: {output_file}")
        print(f"Removed {ws.max_row - len(deet_df)} rows (strikethrough + non-DEET platform)")

    except Exception as e:
        print(f"Error processing DEET sheet: {str(e)}")

# Execute the function
process_deet_sheet()

Processing complete. Output saved to: Outputs\DEET_Analysis_20250807_165549.xlsx
Removed 404 rows (strikethrough + non-DEET platform)


In [25]:
def process_nebula_galaxy():
    try:
        # Read the Nebula Galaxy file
        nebgal_file = file_mapping["Nebula_Galaxy"]["path"]

        # Read Excel file with openpyxl
        wb = openpyxl.load_workbook(nebgal_file)
        ws = wb['NebGal']

        # Create a list to store non-strikethrough row indices
        rows_to_keep = []

        # Check each row for strikethrough formatting
        for row_idx, row in enumerate(ws.rows):
            cell = row[0]
            if not cell.font.strike or cell.value is None:
                rows_to_keep.append(row_idx)
        
        # Read the Excel file into DataFrame
        nebgal_df = pd.read_excel(nebgal_file, sheet_name="NebGal", skiprows=lambda x: x not in rows_to_keep)
        
        # Convert TIME and Event columns to string type first
        for column in ['TIME', 'Event']:
            nebgal_df[column] = nebgal_df[column].astype(str)

        # Process TIME and Event columns
        # Convert values to TRUE/FALSE based on 1/0 values
        for column in ['TIME', 'Event']:
            for idx, value in enumerate(nebgal_df[column]):
                # Convert to string for comparison
                str_value = str(value).strip().lower()
                # Check if value is 1 (or "1")
                if str_value == "1" or str_value == "1.0" or str_value == "true":
                    nebgal_df.at[idx, column] = 'TRUE'
                # Check if value is 0, empty, nan, or "0"
                else:
                    nebgal_df.at[idx, column] = 'FALSE'

        # Add 'Config' column after 'Event' with default FALSE values
        event_idx = nebgal_df.columns.get_loc('Event')
        nebgal_df.insert(event_idx + 1, 'Config', 'FALSE')
        
        # Process Default column
        for idx, row in nebgal_df.iterrows():
            default_val = str(row['Default']).lower()  # Convert to string and lowercase for better matching
            
            # Check for Configuration logging
            if 'reefer configuration' in default_val:
                nebgal_df.at[idx, 'Config'] = 'TRUE'

        alarm_idx = nebgal_df.columns.get_loc('Unit of Measurement')
        new_columns = ['Event Log When Parameter Changes?']

        # Initialize new columns with 'FALSE'
        for idx, col in enumerate(reversed(new_columns)):
            nebgal_df.insert(alarm_idx + 1, col, '')
        
        # Process the Auto log columns that might have YES/NO values
        for column in ['Auto Log at Power ON?', 'Auto Log at Noon (12:05 PM)?']:
            if column in nebgal_df.columns:
                for idx, value in enumerate(nebgal_df[column]):
                    # Convert to string for comparison
                    str_value = str(value).strip().lower()
                    # Check if value is YES
                    if str_value == "yes":
                        nebgal_df.at[idx, column] = 'TRUE'
                    # Check if value is NO or other values
                    elif str_value == "no":
                        nebgal_df.at[idx, column] = 'FALSE'
                    # Leave other values as they are
        
        # Change the Additional Notes column to 'Additional Reqs'
        if 'Additional Notes' in nebgal_df.columns:
            nebgal_df.rename(columns={'Additional Notes': 'Additional Reqs'}, inplace=True)

        # Process Additional Reqs column
        for idx, row in nebgal_df.iterrows():
            additional_reqs = str(row['Additional Reqs']).lower() # Convert to string and lowercase for better matching

            # Check for parameter changes logging
            if any(keyword in additional_reqs for keyword in ['parameter changes', 'param changes', 'value change', 'changes']):
                nebgal_df.at[idx, 'Event Log When Parameter Changes?'] = 'TRUE'

        # Add 'Documentation Notes' column after 'Additional Reqs'
        add_reqs_idx = nebgal_df.columns.get_loc('Additional Reqs')
        nebgal_df.insert(add_reqs_idx + 1, 'Documentation Notes', '')

        # Remove Product, Default, Acronym, and Slot column
        nebgal_df = nebgal_df.drop(['Product', 'Default', 'Acronym', 'Slot'], axis=1)

        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = os.path.join(output_dir, f"Nebula_Galaxy_Analysis{timestamp}.xlsx")
        nebgal_df.to_excel(output_file, index=False)

        print(f"Processing complete. Output saved to: {output_file}")
        print(f"Removed {ws.max_row - len(rows_to_keep)} strikethrough rows.")
    
    except Exception as e:
        print(f"Error processing Nebula-Galaxy sheet: {str(e)}")

# Execute the function
process_nebula_galaxy()

Processing complete. Output saved to: Outputs\Nebula_Galaxy_Analysis20250807_171253.xlsx
Removed 0 strikethrough rows.


In [ ]:
def process_railblazer_sheet():
    try:
        # Read Railblazer Excel file
        rb_file = file_mapping["RB"]["path"]
        
        # Read Excel file with openpyxl
        wb = openpyxl.load_workbook(rb_file)
        ws = wb[file_mapping["RB"]["sheet"]]
        
        # Create a list to store non-strikethrough row indices
        rows_to_keep = []
        
        # Check each row for strikethrough formatting
        for row_idx, row in enumerate(ws.rows):
            cell = row[0]
            if not cell.font.strike or cell.value is None:
                rows_to_keep.append(row_idx)
        
        # Read the Excel file into DataFrame
        rb_df = pd.read_excel(rb_file, sheet_name=file_mapping["RB"]["sheet"], skiprows=lambda x: x not in rows_to_keep)
        
        # Change 'Parameter Name' header to 'User Display Name'
        if 'Parameter Name' in rb_df.columns:
            rb_df.rename(columns={'Parameter Name': 'User Display Name'}, inplace=True)
        
        # Change 'Access Level' header to 'Access Level (Wintrac)'
        if 'Access Level' in rb_df.columns:
            rb_df.rename(columns={'Access Level': 'Access Level (Wintrac)'}, inplace=True)

        # First convert the Event column values to string format and standardize to TRUE/FALSE
        if 'Event' in rb_df.columns:
            # Convert Event column to string type first
            rb_df['Event'] = rb_df['Event'].astype(str)
            
            # Convert values to TRUE/FALSE based on 1/0/True/False values
            for idx, value in enumerate(rb_df['Event']):
                # Convert to string for comparison
                str_value = str(value).strip().lower()
                # Check if value is 1, "1", "true", or True
                if str_value in ['1', '1.0', 'true']:
                    rb_df.at[idx, 'Event'] = 'TRUE'
                # Else set to FALSE
                else:
                    rb_df.at[idx, 'Event'] = 'FALSE'

        # Add 'Config' column after 'Event' with default FALSE values
        event_idx = rb_df.columns.get_loc('Event')
        rb_df.insert(event_idx + 1, 'Config', 'FALSE')

        # Add 'Unit of Measurement' column after 'Event' with default FALSE values
        access_idx = rb_df.columns.get_loc('Access Level (Wintrac)')
        rb_df.insert(access_idx + 1, 'Unit of Measurement', '')

        # Process Default Log column
        if 'Default Log' in rb_df.columns:
            for idx, row in rb_df.iterrows():
                # Check if Default Log is empty or NaN
                if pd.isna(row['Default Log']) or str(row['Default Log']).strip() == '':
                    # If Default Log is empty, set TIME, Event, and Config to empty strings
                    rb_df.at[idx, 'TIME'] = ''
                    rb_df.at[idx, 'Event'] = ''
                    rb_df.at[idx, 'Config'] = ''
                else:
                    # Otherwise, process as normal for non-empty cells
                    default_log = str(row['Default Log']).lower()  # Convert to string and lowercase for better matching

                    # Check for Configuration logging
                    if 'reefer configuration' in default_log:
                        rb_df.at[idx, 'Config'] = 'TRUE'
                        
                    # Check for Timed/Periodic logging
                    if any(keyword in default_log for keyword in ['timed', 'periodic']):
                        rb_df.at[idx, 'TIME'] = 'TRUE'
                    
                    # Check for Event logging
                    if 'event' in default_log:
                        rb_df.at[idx, 'Event'] = 'TRUE'
                
                
        # Add columns after "Event log when alarm code(s) set"
        alarm_col = 'Event log when alarm code(s) set'
        if alarm_col in rb_df.columns:
            alarm_idx = rb_df.columns.get_loc(alarm_col)
            log_columns = [
                'Event Log When Parameter Changes?',
                'Auto Log at Power ON?',
                'Auto Log at Noon (12:05 PM)?'
            ]
            
            # Insert columns after Event log when alarm code(s) set with default '' values
            for idx, col in enumerate(reversed(log_columns)):
                rb_df.insert(alarm_idx + 1, col, '')

            # Process Additional Reqs column
            if 'Additional Reqs' in rb_df.columns:
                for idx, row in rb_df.iterrows():
                    additional_reqs = str(row['Additional Reqs']).lower()  # Convert to string and lowercase for better matching
                    
                    # Check for noon/midday logging
                    if any(keyword in additional_reqs for keyword in ['12:05', 'midday', 'noon', 'middle of day']):
                        rb_df.at[idx, 'Auto Log at Noon (12:05 PM)?'] = 'TRUE'
                    
                    # Check for power on/startup logging
                    if any(keyword in additional_reqs for keyword in ['power on', 'power-on', 'poweron', 'start up', 'startup', 'power cycle']):
                        rb_df.at[idx, 'Auto Log at Power ON?'] = 'TRUE'
                    
                    # Check for parameter/value changes logging
                    if any(keyword in additional_reqs for keyword in ['parameter changes', 'param changes', 'value change', 'state change', 'changes']):
                        rb_df.at[idx, 'Event Log When Parameter Changes?'] = 'TRUE'

        # Remove Platform and Default Log columns
        rb_df = rb_df.drop(['SPN Acronym ', 'Source', 'Default Log'], axis=1)

        # Create output directory and save file
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = os.path.join(output_dir, f'Railblazer_Analysis_{timestamp}.xlsx')
        rb_df.to_excel(output_file, index=False)
        
        print(f"Processing complete. Output saved to: {output_file}")
        print(f"Removed {ws.max_row - len(rows_to_keep)} strikethrough rows")

    except Exception as e:
        print(f"Error processing Railblazer sheet: {str(e)}")

# Execute the function
process_railblazer_sheet()